# USPS Address Standardization with Databricks AI Functions

This notebook demonstrates how to use **`ai_query`** with Databricks Foundation Model APIs
to standardize, validate, and correct US mailing addresses according to
[USPS Publication 28](https://pe.usps.com/text/pub28/welcome.htm) rules.

## Why `ai_query`?

| Approach | Strengths | Limitations |
|----------|-----------|-------------|
| **`ai_query`** | Full prompt control, structured JSON output, reasoning for corrections | Cost per row, latency |
| `ai_extract` | Fast component extraction | No validation or correction capability |
| `ai_classify` | Quality scoring | Categorical only — can't fix addresses |
| Python UDF (`usaddress`) | Fast parsing | No actual USPS validation or correction |

**`ai_query` is the recommended native approach** because address standardization requires
*reasoning* — fixing typos, expanding/contracting abbreviations, inferring missing components,
and applying USPS formatting rules. The other AI functions are useful supplements but cannot
perform the full standardization task.

## Step 1: Generate Synthetic Address Data

We create a mix of address scenarios: clean addresses, common errors, missing components,
abbreviation inconsistencies, and formatting problems.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Synthetic addresses with various quality issues
address_data = [
    # --- Clean addresses (should pass through with minor formatting) ---
    (1,  "100 Main Street",        "Suite 200",  "San Francisco",  "CA",    "94105",  "Clean - standard format"),
    (2,  "1600 Pennsylvania Ave NW", None,        "Washington",     "DC",    "20500",  "Clean - well-known address"),
    (3,  "350 Fifth Avenue",       "Floor 34",   "New York",       "NY",    "10118",  "Clean - Empire State Building"),

    # --- Abbreviation issues (USPS Pub 28 requires standard abbreviations) ---
    (4,  "742 Evergreen Terrace",  None,         "Springfield",    "Illinois", "62704", "State not abbreviated"),
    (5,  "123 North Main Street",  "Apartment 4B", "Denver",      "CO",    "80202",  "Street/unit not abbreviated"),
    (6,  "456 West Elm Boulevard", "Building C", "Austin",         "Texas", "78701",  "State and suffix not abbreviated"),

    # --- Typos and misspellings ---
    (7,  "789 Brodway",           None,          "New Yrok",       "NY",    "10003",  "Typos in street and city"),
    (8,  "321 Mian Stret",        "Ste 100",    "Sacremento",     "CA",    "95814",  "Multiple misspellings"),
    (9,  "555 Pensilvania Ave",   None,          "Philidelphia",   "PA",    "19103",  "City and street misspelled"),

    # --- Missing components ---
    (10, "1000 Oak Drive",         None,         "Portland",       None,    "97201",  "Missing state"),
    (11, "200 Lake Shore Drive",   "Unit 5A",    "Chicago",        "IL",    None,     "Missing ZIP code"),
    (12, "88 Colin P Kelly Jr St", None,         None,             "CA",    "94107",  "Missing city (GitHub HQ area)"),

    # --- Incorrect ZIP codes ---
    (13, "1 Infinite Loop",        None,         "Cupertino",      "CA",    "90210",  "Wrong ZIP (Beverly Hills ZIP)"),
    (14, "410 Terry Ave N",        None,         "Seattle",        "WA",    "10001",  "Wrong ZIP (NYC ZIP)"),
    (15, "1 Hacker Way",           None,         "Menlo Park",     "CA",    "20500",  "Wrong ZIP (DC ZIP)"),

    # --- Formatting chaos ---
    (16, "  123   MAIN   ST  ",   " apt 2b ",   "  LOS ANGELES ", "ca",    "90012 ", "Extra spaces, wrong case"),
    (17, "456 elm st.",            "ste. 300",   "houston",        "tx",    "77001",  "All lowercase, periods"),
    (18, "789 OAK BOULEVARD, SUITE 500", None,   "MIAMI",         "FL",    "33101",  "All caps, comma in address"),

    # --- Directional and numeric issues ---
    (19, "100 North West 1st Street", None,      "Miami",          "FL",    "33128",  "Directional should be NW"),
    (20, "2000 South East Main Ave", "Bldg 3",  "Portland",       "OR",    "97214",  "Directional should be SE"),

    # --- Secondary unit designator issues ---
    (21, "500 Market Street",      "Number 401", "San Francisco",  "CA",    "94105",  "Non-standard unit designator"),
    (22, "750 7th Avenue",         "Room 2401",  "New York",       "NY",    "10019",  "Room vs Ste/Rm"),
    (23, "300 E Main St",          "#12",        "Lexington",      "KY",    "40507",  "Pound sign unit designator"),

    # --- PO Box and Rural Routes ---
    (24, "Post Office Box 12345",  None,         "Anytown",        "KS",    "66002",  "PO Box not abbreviated"),
    (25, "Rural Route 2 Box 150",  None,         "Springfield",    "MO",    "65801",  "Rural route not abbreviated"),

    # --- Complex / edge cases ---
    (26, "1-800 Corporate Drive",  "Mailstop 4-2", "Raleigh",     "NC",    "27607",  "Unusual street number format"),
    (27, "One Microsoft Way",      None,         "Redmond",        "WA",    "98052",  "Written-out number"),
    (28, "1959 NE Pacific St",     None,         "Seattle",        "WA",    "98195",  "UW Medical Center"),
    (29, "1600 Amphitheatre Pkwy", None,         "Mountain View",  "CA",    "94043",  "Abbreviation Pkwy vs Parkway"),
    (30, "233 S Wacker Dr",        "Fl 103",     "Chicago",        "IL",    "60606",  "Willis/Sears Tower - Fl vs Floor"),
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("address_line_1", StringType(), True),
    StructField("address_line_2", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("issue_description", StringType(), True),
])

df_raw = spark.createDataFrame(address_data, schema)
df_raw.createOrReplaceTempView("raw_addresses")

display(df_raw)

## Step 2: Standardize Addresses with `ai_query`

We use `ai_query` to call a Foundation Model endpoint with a detailed USPS standardization prompt.
The model returns structured JSON with the corrected address and confidence metadata.

In [0]:
%sql
-- Create the standardized output using ai_query
-- This calls the Foundation Model API with USPS standardization rules

CREATE OR REPLACE TEMPORARY VIEW standardized_addresses AS
SELECT
  id,
  address_line_1 AS original_address_line_1,
  address_line_2 AS original_address_line_2,
  city AS original_city,
  state AS original_state,
  zip_code AS original_zip_code,
  issue_description,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'You are a USPS address standardization expert. Standardize the following address according to USPS Publication 28 rules.\n\n',
      'RULES:\n',
      '1. Use standard USPS abbreviations for street suffixes (ST, AVE, BLVD, DR, LN, CT, PL, RD, WAY, CIR, PKWY, TER)\n',
      '2. Use 2-letter state abbreviations (CA, NY, TX, etc.)\n',
      '3. Abbreviate directionals (N, S, E, W, NE, NW, SE, SW)\n',
      '4. Use standard secondary unit designators (STE, APT, UNIT, FL, RM, BLDG, # is acceptable)\n',
      '5. Uppercase all text\n',
      '6. Remove extra spaces, periods, and commas\n',
      '7. PO BOX format: PO BOX [number]\n',
      '8. Rural routes: RR [number] BOX [number]\n',
      '9. Fix obvious typos and misspellings in street names and city names\n',
      '10. If ZIP code appears incorrect for the city/state, provide the correct ZIP if you are confident, otherwise keep the original and flag it\n',
      '11. If a component is missing and you can confidently infer it, add it; otherwise leave it null and flag it\n\n',
      'ADDRESS TO STANDARDIZE:\n',
      'Line 1: ', COALESCE(address_line_1, ''), '\n',
      'Line 2: ', COALESCE(address_line_2, ''), '\n',
      'City: ', COALESCE(city, ''), '\n',
      'State: ', COALESCE(state, ''), '\n',
      'ZIP: ', COALESCE(zip_code, ''), '\n\n',
      'Respond with ONLY a JSON object (no markdown, no code fences) with these fields:\n',
      '{\n',
      '  "address_line_1": "standardized line 1",\n',
      '  "address_line_2": "standardized line 2 or null",\n',
      '  "city": "standardized city",\n',
      '  "state": "2-letter state code",\n',
      '  "zip_code": "5-digit or ZIP+4",\n',
      '  "confidence": "HIGH, MEDIUM, or LOW",\n',
      '  "changes_made": ["list of changes applied"],\n',
      '  "warnings": ["list of any issues or uncertainties"]\n',
      '}'
    )
  ) AS ai_response
FROM raw_addresses

## Step 3: Parse the AI Response into Structured Columns

In [0]:
%sql
-- Parse the JSON response into individual columns for analysis

CREATE OR REPLACE TEMPORARY VIEW parsed_results AS
SELECT
  id,
  original_address_line_1,
  original_address_line_2,
  original_city,
  original_state,
  original_zip_code,
  issue_description,
  ai_response,
  get_json_object(ai_response, '$.address_line_1') AS std_address_line_1,
  get_json_object(ai_response, '$.address_line_2') AS std_address_line_2,
  get_json_object(ai_response, '$.city') AS std_city,
  get_json_object(ai_response, '$.state') AS std_state,
  get_json_object(ai_response, '$.zip_code') AS std_zip_code,
  get_json_object(ai_response, '$.confidence') AS confidence,
  get_json_object(ai_response, '$.changes_made') AS changes_made,
  get_json_object(ai_response, '$.warnings') AS warnings
FROM standardized_addresses

In [0]:
%sql
-- Display the standardized results with a side-by-side comparison
SELECT
  id,
  issue_description,
  -- Original
  CONCAT_WS(', ',
    original_address_line_1,
    original_address_line_2,
    original_city,
    original_state,
    original_zip_code
  ) AS original_full_address,
  -- Standardized
  CONCAT_WS(', ',
    std_address_line_1,
    std_address_line_2,
    std_city,
    std_state,
    std_zip_code
  ) AS standardized_full_address,
  confidence,
  changes_made,
  warnings
FROM parsed_results
ORDER BY id

## Step 4: Quality Analysis Dashboard

In [0]:
%sql
-- Confidence distribution
SELECT
  confidence,
  COUNT(*) AS address_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM parsed_results), 1) AS percentage
FROM parsed_results
GROUP BY confidence
ORDER BY
  CASE confidence
    WHEN 'HIGH' THEN 1
    WHEN 'MEDIUM' THEN 2
    WHEN 'LOW' THEN 3
    ELSE 4
  END

In [0]:
%sql
-- Identify addresses that had changes vs. those that were already correct
SELECT
  CASE
    WHEN changes_made = '[]' OR changes_made IS NULL THEN 'No Changes Needed'
    ELSE 'Changes Applied'
  END AS change_status,
  COUNT(*) AS address_count
FROM parsed_results
GROUP BY change_status

In [0]:
%sql
-- Show addresses with warnings (potential issues that need human review)
SELECT
  id,
  issue_description,
  CONCAT_WS(', ', original_address_line_1, original_city, original_state, original_zip_code) AS original,
  CONCAT_WS(', ', std_address_line_1, std_city, std_state, std_zip_code) AS standardized,
  confidence,
  warnings
FROM parsed_results
WHERE warnings != '[]' AND warnings IS NOT NULL
ORDER BY
  CASE confidence
    WHEN 'LOW' THEN 1
    WHEN 'MEDIUM' THEN 2
    WHEN 'HIGH' THEN 3
  END

## Step 5: Batch Processing Pattern for Large Datasets

For production workloads with millions of rows, use these optimization strategies:
- **Partition the data** and process in batches to manage API throughput
- **Cache results** to avoid re-processing unchanged addresses
- **Filter first** — only send addresses that actually need standardization
- **Use `ai_query` with `modelParameters`** to control temperature (set to 0 for deterministic output)

In [0]:
%sql
-- Production pattern: Only standardize addresses that appear to need correction
-- This pre-filter reduces API calls significantly

SELECT
  id,
  address_line_1,
  city,
  state,
  zip_code,
  CASE
    WHEN state IS NULL OR LENGTH(state) != 2 THEN 'Missing/invalid state'
    WHEN zip_code IS NULL OR LENGTH(TRIM(zip_code)) != 5 THEN 'Missing/invalid ZIP'
    WHEN address_line_1 != UPPER(address_line_1) THEN 'Not uppercased'
    WHEN address_line_1 LIKE '%.%' THEN 'Contains periods'
    WHEN address_line_1 LIKE '%  %' THEN 'Extra spaces'
    WHEN city IS NULL THEN 'Missing city'
    ELSE 'Potentially clean'
  END AS pre_filter_reason
FROM raw_addresses
ORDER BY
  CASE
    WHEN state IS NULL OR LENGTH(state) != 2 THEN 1
    WHEN zip_code IS NULL OR LENGTH(TRIM(zip_code)) != 5 THEN 2
    WHEN city IS NULL THEN 3
    ELSE 10
  END

## Step 6: Alternative Approach — `ai_extract` for Component Parsing

While `ai_query` is best for full standardization, `ai_extract` is useful for
quickly parsing unstructured single-line addresses into components.

In [0]:
%sql
-- Use ai_extract to parse single-line addresses into components
-- This is useful as a preprocessing step

SELECT
  address_text,
  ai_extract(address_text, ARRAY('street_address', 'city', 'state', 'zip_code')) AS extracted
FROM (
  VALUES
    ('123 Main St, San Francisco, CA 94105'),
    ('456 Elm Boulevard Suite 300 Houston TX 77001'),
    ('PO Box 789, Anytown KS 66002'),
    ('1600 Pennsylvania Ave NW Washington DC 20500')
) AS t(address_text)

## Step 7: Combine Approaches — Extract + Standardize Pipeline

For messy single-line addresses, combine `ai_extract` (parsing) with `ai_query` (standardization).

In [0]:
%sql
-- Pipeline: Extract components, then standardize
WITH extracted AS (
  SELECT
    address_text,
    ai_extract(address_text, ARRAY('street_address', 'city', 'state', 'zip_code')) AS components
  FROM (
    VALUES
      ('789 brodway new yrok ny 10003'),
      ('321 mian stret ste 100 sacremento california 95814'),
      ('100 north west 1st street miami fl 33128')
  ) AS t(address_text)
)
SELECT
  address_text AS original_single_line,
  components,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Standardize this US address per USPS Publication 28 rules. ',
      'Uppercase all text, use standard abbreviations, fix typos. ',
      'Return ONLY a JSON object with: address_line_1, city, state, zip_code, confidence.\n\n',
      'Address: ', address_text
    )
  ) AS standardized
FROM extracted

## Summary & Recommendations

### When to use each approach

| Scenario | Recommended Approach |
|----------|---------------------|
| Full address standardization & correction | **`ai_query`** with detailed USPS prompt |
| Parse single-line addresses into components | **`ai_extract`** |
| Classify address quality (good/bad/needs-review) | **`ai_classify`** |
| High-volume batch processing | **`ai_query`** with pre-filtering + partitioning |
| Pipeline: unstructured → structured → standardized | **`ai_extract`** → **`ai_query`** |

### Production considerations

1. **Cost**: Each `ai_query` call consumes tokens. Pre-filter to only process addresses that need correction.
2. **Throughput**: Foundation Model APIs have rate limits. Partition large datasets and process in batches.
3. **Determinism**: Set temperature to 0 for consistent results across runs.
4. **Validation**: For mission-critical use cases, consider a second pass with a different model or USPS CASS-certified API for final verification.
5. **Caching**: Store standardized results and only re-process when source addresses change.